# 🌐 app.py — Farmer Mobile Web App (Flask)
**KrishiAI · IT-A · 2024-25**

This notebook writes `app.py` to disk and explains how to run it.  
`app.py` is a Flask server that serves both:
- The **farmer mobile web UI** (Kannada + English, camera upload, WhatsApp share)
- The **`/predict` API** (accepts image → returns JSON diagnosis)

### How it works
```
Farmer's phone  ──POST /predict──►  Flask server  ──►  TFLite INT8 model
     ◄─────────────── JSON result ───────────────── disease + Kannada alert
```

> ⚠️ **Run `05_step4_export_and_infer.ipynb` first** — the server loads the TFLite model.  
> ℹ️ Flask cannot run inside a Jupyter cell directly. This notebook writes the file  
> and shows you two ways to launch it.

In [ ]:
!pip install flask flask-cors pillow --quiet
print("✅ Flask ready")

In [ ]:
# ── config.py is the shared settings file — paste it here so this notebook
# ── is self-contained. All other notebooks also start with this block.
# =============================================================================
# config.py — KrishiAI Project Configuration
# All hyperparameters, paths, and class definitions in ONE place.
# Change values here; the rest of the scripts pick them up automatically.
# =============================================================================

import os

# ── PATHS ──────────────────────────────────────────────────────────────────────
BASE_DIR        = os.getcwd()   # notebook working directory
DATA_RAW_DIR    = os.path.join(BASE_DIR, "data", "raw")          # downloaded zips
DATA_CLEAN_DIR  = os.path.join(BASE_DIR, "data", "cleaned")      # after cleaning
DATA_PROC_DIR   = os.path.join(BASE_DIR, "data", "processed")    # train/val/test splits
MODELS_DIR      = os.path.join(BASE_DIR, "models")               # saved .keras / .h5
TFLITE_DIR      = os.path.join(BASE_DIR, "tflite")               # exported .tflite
LOGS_DIR        = os.path.join(BASE_DIR, "logs")                 # TensorBoard / CSVs
PLOTS_DIR       = os.path.join(BASE_DIR, "plots")                # confusion matrix, curves

for d in [DATA_RAW_DIR, DATA_CLEAN_DIR, DATA_PROC_DIR,
          MODELS_DIR, TFLITE_DIR, LOGS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── DATASET ───────────────────────────────────────────────────────────────────
# PlantVillage is available via tensorflow_datasets (no Kaggle key needed).
# We filter it down to only classes relevant to Udupi crops.
# The keys below MUST match the class folder names in PlantVillage exactly.

DISEASE_CLASSES = {
    # ── Rice ──────────────────────────────────────────────────────────────────
    "Rice___Leaf_blast":          "Rice Blast",            # Magnaporthe oryzae
    "Rice___Bacterial_leaf_blight": "Bacterial Leaf Blight", # Xanthomonas oryzae
    "Rice___Brown_spot":          "Brown Spot",            # Bipolaris oryzae
    "Rice___healthy":             "Healthy Rice",

    # ── Coconut / proxy (PlantVillage doesn't have coconut; we use the
    #    closest abiotic-stress proxy — corn with nutrient deficiency)
    "Corn_(maize)___Northern_Leaf_Blight": "Leaf Blight (proxy)",
    "Corn_(maize)___healthy":              "Healthy Crop (proxy)",

    # ── Tomato (used as augmentation source for fungal patterns) ──────────────
    "Tomato___Late_blight":       "Fungal Blight (proxy)",
    "Tomato___healthy":           "Healthy Crop (proxy)",
}

# Friendly short labels for plotting (must stay in same order as CLASS_NAMES list)
CLASS_NAMES = [
    "Rice Blast",
    "Bacterial Blight",
    "Brown Spot",
    "Healthy Rice",
    "Leaf Blight",
    "Fungal Blight",
    "Healthy Crop",
]

NUM_CLASSES = len(CLASS_NAMES)

# ── MODEL ─────────────────────────────────────────────────────────────────────
IMAGE_SIZE   = (224, 224)   # EfficientNetB0 native input size
BATCH_SIZE   = 32
EPOCHS_FROZEN  = 10        # Phase 1: train only the classification head
EPOCHS_FINETUNE = 20       # Phase 2: unfreeze top layers and fine-tune
LEARNING_RATE_HEAD    = 1e-3
LEARNING_RATE_FINETUNE = 1e-5
DROPOUT_RATE = 0.3
L2_LAMBDA    = 1e-4

# ── DATA SPLIT ────────────────────────────────────────────────────────────────
TRAIN_SPLIT = 0.70
VAL_SPLIT   = 0.15
TEST_SPLIT  = 0.15
RANDOM_SEED = 42

# ── DATA CLEANING THRESHOLDS ──────────────────────────────────────────────────
MIN_IMAGE_SIZE_BYTES = 2_000        # discard images smaller than 2 KB (corrupt)
MIN_PIXEL_DIMENSION  = 64           # discard images narrower/shorter than 64px
MAX_BLUR_VARIANCE    = 10.0         # discard images with Laplacian variance < 10
                                    # (too blurry to be useful)

# ── AUGMENTATION ──────────────────────────────────────────────────────────────
# Tuned for monsoon / field conditions in Karnataka
AUGMENT_CONFIG = {
    "rotation_range":     20,        # ±20° for handheld phone shots
    "zoom_range":         0.2,       # simulate distance variation
    "horizontal_flip":    True,
    "brightness_range":   (0.7, 1.3),# monsoon overcast → harsh afternoon sun
    "width_shift_range":  0.1,
    "height_shift_range": 0.1,
    "shear_range":        0.1,
    "fill_mode":          "reflect",
}

# ── TFLITE EXPORT ─────────────────────────────────────────────────────────────
TFLITE_MODEL_NAME  = "krishiai_efficientnetb0_int8.tflite"
TFLITE_TARGET_MB   = 4.0   # must stay under 4 MB for ₹8k Android phones

# ── KANNADA MESSAGES ──────────────────────────────────────────────────────────
# Shown in the inference output alongside the English diagnosis
KANNADA_ALERTS = {
    "Rice Blast":          "ಭತ್ತದ ಬ್ಲಾಸ್ಟ್ ರೋಗ ಪತ್ತೆ. ಟ್ರೈಸೈಕ್ಲಾಜ಼ೋಲ್ ತಕ್ಷಣ ಸಿಂಪಡಿಸಿ.",
    "Bacterial Blight":    "ಬ್ಯಾಕ್ಟೀರಿಯಲ್ ಎಲೆ ಒಣಗುವಿಕೆ. ಕಾಪರ್ ಆಕ್ಸಿಕ್ಲೋರೈಡ್ ಬಳಸಿ.",
    "Brown Spot":          "ಕಂದು ಚುಕ್ಕೆ ರೋಗ. ಮ್ಯಾಂಕೋಜ಼ೆಬ್ ಬಳಸಿ.",
    "Healthy Rice":        "ಬೆಳೆ ಆರೋಗ್ಯಕರವಾಗಿದೆ. ಮೇಲ್ವಿಚಾರಣೆ ಮುಂದುವರಿಸಿ.",
    "Leaf Blight":         "ಎಲೆ ಒಣಗುವ ರೋಗ. KVK ಸಂಪರ್ಕಿಸಿ.",
    "Fungal Blight":       "ಶಿಲೀಂಧ್ರ ರೋಗ ಪತ್ತೆ. ಶಿಲೀಂಧ್ರನಾಶಕ ಸಿಂಪಡಿಸಿ.",
    "Healthy Crop":        "ಬೆಳೆ ಆರೋಗ್ಯಕರ. ನಿಯಮಿತ ತಪಾಸಣೆ ಮಾಡಿ.",
}

TREATMENT_MAP = {
    "Rice Blast":       ["Tricyclazole 75WP @ 0.6 g/L", "Isoprothiolane 40EC @ 1.5 mL/L", "Avoid excess nitrogen", "Drain stagnant water"],
    "Bacterial Blight": ["Copper Oxychloride 50WP @ 3 g/L", "Streptocycline @ 0.5 g/L", "Avoid flood irrigation", "Remove infected debris"],
    "Brown Spot":       ["Mancozeb 75WP @ 2.5 g/L", "Propiconazole 25EC @ 1 mL/L", "Correct potassium deficiency"],
    "Healthy Rice":     ["Continue regular monitoring", "Preventive fungicide (optional at tillering)"],
    "Leaf Blight":      ["Copper-based fungicide", "Consult KVK Udupi: 0820-2520842"],
    "Fungal Blight":    ["Mancozeb + Carbendazim combo", "Improve field drainage"],
    "Healthy Crop":     ["Continue monitoring", "Check weekly during monsoon"],
}


## 💾 Write app.py to Disk

In [ ]:
# Write app.py to the working directory
app_code = '# =============================================================================\n# app.py  —  KrishiAI Flask Backend\n#\n# Serves the farmer mobile web app AND the prediction API.\n# The TFLite model runs entirely on the server, so the farmer\'s ₹8k phone\n# only needs to send a photo — no ML library needed on-device.\n#\n# Routes:\n#   GET  /                    → Farmer mobile web app (farmer.html)\n#   POST /predict             → Accept image, return JSON diagnosis\n#   GET  /history             → Last 50 scans for this farmer (session-based)\n#   GET  /health              → Server health check\n#\n# Run:\n#   pip install flask flask-cors pillow\n#   python app.py\n#\n# Then open http://localhost:5000 on any phone on the same WiFi.\n# For public access: use ngrok → ngrok http 5000\n# =============================================================================\n\nimport os\nimport sys\nimport json\nimport uuid\nimport time\nimport base64\nimport logging\nfrom datetime import datetime\nfrom pathlib import Path\nfrom io import BytesIO\n\nimport numpy as np\nfrom PIL import Image, ImageOps\nfrom flask import Flask, request, jsonify, render_template_string, session\nfrom flask_cors import CORS\n\n# Add project root so we can import config + inference engine\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nfrom config import (\n    TFLITE_DIR, TFLITE_MODEL_NAME, IMAGE_SIZE, NUM_CLASSES,\n    CLASS_NAMES, KANNADA_ALERTS, TREATMENT_MAP, LOGS_DIR\n)\n\n# ── Logging ───────────────────────────────────────────────────────────────────\nlogging.basicConfig(\n    level=logging.INFO,\n    format="%(asctime)s  %(levelname)s  %(message)s",\n    handlers=[\n        logging.StreamHandler(),\n        logging.FileHandler(os.path.join(LOGS_DIR, "server.log"), encoding="utf-8"),\n    ]\n)\nlog = logging.getLogger("KrishiAI")\n\n# ── Flask app ──────────────────────────────────────────────────────────────────\napp = Flask(__name__)\napp.secret_key = "krishiai-udupi-2025"   # change in production\nCORS(app)\n\n# Scan history stored in memory (use SQLite/PostgreSQL for production)\nSCAN_HISTORY = []\nMAX_HISTORY  = 200\n\n# ── Load TFLite inference engine once at startup ───────────────────────────────\ntry:\n    from step4_export_and_infer import KrishiAIInference\n    ENGINE = KrishiAIInference()\n    MODEL_READY = True\n    log.info("✅ TFLite model loaded successfully")\nexcept Exception as e:\n    log.warning(f"⚠️  TFLite model not found ({e}). Running in DEMO mode.")\n    ENGINE      = None\n    MODEL_READY = False\n\n\n# ──────────────────────────────────────────────────────────────────────────────\n# MOCK ENGINE for demo / before training is complete\n# ──────────────────────────────────────────────────────────────────────────────\n\nDEMO_RESULTS = [\n    {\n        "disease": "Rice Blast",\n        "confidence": 0.93,\n        "confidence_pct": "93.0%",\n        "risk_level": "HIGH",\n        "kannada_alert": "ಭತ್ತದ ಬ್ಲಾಸ್ಟ್ ರೋಗ ಪತ್ತೆ. ಟ್ರೈಸೈಕ್ಲಾಜ಼ೋಲ್ ತಕ್ಷಣ ಸಿಂಪಡಿಸಿ.",\n        "treatment": ["Tricyclazole 75WP @ 0.6 g/L", "Isoprothiolane 40EC @ 1.5 mL/L", "Avoid excess nitrogen", "Drain stagnant water"],\n        "top_k": [\n            {"disease": "Rice Blast",         "confidence": 0.93},\n            {"disease": "Brown Spot",         "confidence": 0.05},\n            {"disease": "Bacterial Blight",   "confidence": 0.02},\n        ],\n        "inference_ms": 312.0,\n    },\n    {\n        "disease": "Healthy Rice",\n        "confidence": 0.96,\n        "confidence_pct": "96.0%",\n        "risk_level": "LOW",\n        "kannada_alert": "ಬೆಳೆ ಆರೋಗ್ಯಕರವಾಗಿದೆ. ಮೇಲ್ವಿಚಾರಣೆ ಮುಂದುವರಿಸಿ.",\n        "treatment": ["Continue regular monitoring", "Preventive fungicide (optional)"],\n        "top_k": [\n            {"disease": "Healthy Rice",       "confidence": 0.96},\n            {"disease": "Rice Blast",         "confidence": 0.03},\n            {"disease": "Brown Spot",         "confidence": 0.01},\n        ],\n        "inference_ms": 287.0,\n    },\n    {\n        "disease": "Bacterial Blight",\n        "confidence": 0.87,\n        "confidence_pct": "87.0%",\n        "risk_level": "MODERATE",\n        "kannada_alert": "ಬ್ಯಾಕ್ಟೀರಿಯಲ್ ಎಲೆ ಒಣಗುವಿಕೆ. ಕಾಪರ್ ಆಕ್ಸಿಕ್ಲೋರೈಡ್ ಬಳಸಿ.",\n        "treatment": ["Copper Oxychloride 50WP @ 3 g/L", "Streptocycline @ 0.5 g/L", "Avoid flood irrigation"],\n        "top_k": [\n            {"disease": "Bacterial Blight",   "confidence": 0.87},\n            {"disease": "Healthy Rice",       "confidence": 0.08},\n            {"disease": "Rice Blast",         "confidence": 0.05},\n        ],\n        "inference_ms": 298.0,\n    },\n]\n_demo_counter = 0\n\n\ndef run_inference(pil_image: Image.Image) -> dict:\n    """Run real or demo inference on a PIL image."""\n    global _demo_counter\n\n    # Resize image to model input size\n    pil_image = ImageOps.fit(pil_image.convert("RGB"), IMAGE_SIZE, Image.LANCZOS)\n    img_array = np.array(pil_image)   # (224, 224, 3) uint8\n\n    if MODEL_READY and ENGINE is not None:\n        result = ENGINE.predict(img_array)\n    else:\n        # Demo mode — cycle through sample results\n        result = DEMO_RESULTS[_demo_counter % len(DEMO_RESULTS)].copy()\n        _demo_counter += 1\n        result["inference_ms"] = round(200 + np.random.uniform(0, 200), 1)\n        result["demo_mode"] = True\n\n    return result\n\n\n# ──────────────────────────────────────────────────────────────────────────────\n# API ROUTES\n# ──────────────────────────────────────────────────────────────────────────────\n\n@app.route("/health")\ndef health():\n    return jsonify({\n        "status":      "ok",\n        "model_ready": MODEL_READY,\n        "mode":        "live" if MODEL_READY else "demo",\n        "timestamp":   datetime.now().isoformat(),\n    })\n\n\n@app.route("/predict", methods=["POST"])\ndef predict():\n    """\n    Accept a crop photo and return disease diagnosis.\n\n    Accepts:\n      • multipart/form-data  with field "image"\n      • application/json     with field "image_b64" (base64-encoded JPEG/PNG)\n\n    Returns JSON:\n      {\n        "scan_id":        "abc123",\n        "disease":        "Rice Blast",\n        "confidence_pct": "93.0%",\n        "risk_level":     "HIGH",\n        "kannada_alert":  "ಭತ್ತದ ...",\n        "treatment":      ["Tricyclazole 75WP ...", ...],\n        "top_k":          [...],\n        "inference_ms":   312.0,\n        "timestamp":      "2026-03-22T08:14:00",\n        "demo_mode":      false,\n      }\n    """\n    t_start = time.perf_counter()\n\n    # ── Parse image ────────────────────────────────────────────────────────────\n    pil_image = None\n\n    if "image" in request.files:\n        # multipart upload (standard HTML form or camera capture)\n        file = request.files["image"]\n        if file.filename == "":\n            return jsonify({"error": "No file selected"}), 400\n        try:\n            pil_image = Image.open(file.stream)\n        except Exception as e:\n            return jsonify({"error": f"Cannot read image: {e}"}), 400\n\n    elif request.is_json and "image_b64" in request.json:\n        # Base64 upload (from JS fetch with canvas snapshot)\n        try:\n            b64_data = request.json["image_b64"]\n            if "," in b64_data:\n                b64_data = b64_data.split(",")[1]   # strip data:image/jpeg;base64,\n            img_bytes = base64.b64decode(b64_data)\n            pil_image = Image.open(BytesIO(img_bytes))\n        except Exception as e:\n            return jsonify({"error": f"Cannot decode base64 image: {e}"}), 400\n    else:\n        return jsonify({"error": "Send image as multipart \'image\' field or JSON \'image_b64\'"}), 400\n\n    # ── Run inference ──────────────────────────────────────────────────────────\n    try:\n        result = run_inference(pil_image)\n    except Exception as e:\n        log.error(f"Inference error: {e}")\n        return jsonify({"error": f"Inference failed: {e}"}), 500\n\n    # ── Build response ─────────────────────────────────────────────────────────\n    scan_id   = str(uuid.uuid4())[:8].upper()\n    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")\n\n    # Farmer name from form (optional)\n    farmer_name = request.form.get("farmer_name", "") or (\n        request.json.get("farmer_name", "") if request.is_json else ""\n    )\n    crop_type   = request.form.get("crop_type", "paddy") or (\n        request.json.get("crop_type", "paddy") if request.is_json else "paddy"\n    )\n\n    response = {\n        "scan_id":        scan_id,\n        "timestamp":      timestamp,\n        "farmer_name":    farmer_name,\n        "crop_type":      crop_type,\n        **result,\n        "total_ms": round((time.perf_counter() - t_start) * 1000, 1),\n    }\n\n    # ── Store in history ───────────────────────────────────────────────────────\n    SCAN_HISTORY.insert(0, {\n        "scan_id":       scan_id,\n        "timestamp":     timestamp,\n        "farmer_name":   farmer_name,\n        "crop_type":     crop_type,\n        "disease":       result["disease"],\n        "confidence_pct": result["confidence_pct"],\n        "risk_level":    result["risk_level"],\n    })\n    if len(SCAN_HISTORY) > MAX_HISTORY:\n        SCAN_HISTORY.pop()\n\n    log.info(f"Scan {scan_id} | {result[\'disease\']} | {result[\'confidence_pct\']} | {farmer_name or \'anon\'}")\n\n    return jsonify(response)\n\n\n@app.route("/history")\ndef history():\n    """Return the last 50 scans."""\n    return jsonify(SCAN_HISTORY[:50])\n\n\n@app.route("/")\ndef farmer_app():\n    """Serve the farmer mobile web app."""\n    return render_template_string(FARMER_HTML)\n\n\n# ──────────────────────────────────────────────────────────────────────────────\n# FARMER MOBILE WEB APP (inline HTML — no separate file needed)\n# Designed for low-end Android phones, bad network, one-handed use.\n# ──────────────────────────────────────────────────────────────────────────────\n\nFARMER_HTML = r"""<!DOCTYPE html>\n<html lang="kn">\n<head>\n<meta charset="UTF-8">\n<meta name="viewport" content="width=device-width, initial-scale=1, maximum-scale=1, user-scalable=no">\n<meta name="theme-color" content="#1a3d1a">\n<meta name="mobile-web-app-capable" content="yes">\n<meta name="apple-mobile-web-app-capable" content="yes">\n<title>KrishiAI — ಬೆಳೆ ರೋಗ ಪತ್ತೆ</title>\n<link rel="preconnect" href="https://fonts.googleapis.com">\n<link href="https://fonts.googleapis.com/css2?family=Baloo+2:wght@400;500;600;700;800&family=Noto+Sans+Kannada:wght@400;500;600;700&display=swap" rel="stylesheet">\n\n<style>\n:root {\n  --green-deep:   #0d2b0d;\n  --green-dark:   #1a3d1a;\n  --green-mid:    #2d6a2d;\n  --green-main:   #3d8b3d;\n  --green-light:  #5aaf5a;\n  --green-pale:   #c8e6c9;\n  --gold:         #f4a800;\n  --gold-light:   #fff3cd;\n  --red:          #c62828;\n  --red-light:    #ffebee;\n  --orange:       #e65100;\n  --orange-light: #fff3e0;\n  --text-dark:    #1a2e1a;\n  --text-mid:     #3d5a3d;\n  --text-light:   #6b8f6b;\n  --white:        #ffffff;\n  --surface:      #f4faf4;\n  --border:       #c3e6c3;\n}\n\n*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }\n\nhtml { font-size: 16px; -webkit-text-size-adjust: 100%; }\n\nbody {\n  font-family: \'Baloo 2\', \'Noto Sans Kannada\', sans-serif;\n  background: var(--green-deep);\n  color: var(--text-dark);\n  min-height: 100vh;\n  overflow-x: hidden;\n}\n\n/* ── TOP BAR ── */\n.topbar {\n  background: var(--green-dark);\n  padding: 14px 20px 12px;\n  display: flex; align-items: center; gap: 12px;\n  position: sticky; top: 0; z-index: 100;\n  border-bottom: 2px solid var(--green-mid);\n}\n.topbar-logo { font-size: 26px; line-height: 1; }\n.topbar-name { font-size: 20px; font-weight: 800; color: #7ed87e; letter-spacing: -0.3px; }\n.topbar-sub  { font-size: 11px; color: #5a8a5a; font-family: \'Noto Sans Kannada\', sans-serif; }\n.topbar-right { margin-left: auto; }\n.mode-pill {\n  font-size: 10px; font-weight: 700; padding: 3px 10px;\n  border-radius: 20px; background: rgba(90,175,90,0.2);\n  color: #7ed87e; border: 1px solid rgba(90,175,90,0.3);\n  letter-spacing: 0.5px;\n}\n\n/* ── SCROLL CONTAINER ── */\n.scroll-wrap {\n  background: var(--surface);\n  min-height: calc(100vh - 58px);\n  padding-bottom: 40px;\n}\n\n/* ── HERO BAND ── */\n.hero {\n  background: linear-gradient(135deg, var(--green-dark) 0%, var(--green-mid) 100%);\n  padding: 24px 20px 28px;\n  text-align: center;\n  position: relative; overflow: hidden;\n}\n.hero::before {\n  content: \'\';\n  position: absolute; inset: 0;\n  background: url("data:image/svg+xml,%3Csvg width=\'60\' height=\'60\' viewBox=\'0 0 60 60\' xmlns=\'http://www.w3.org/2000/svg\'%3E%3Cg fill=\'none\' fill-rule=\'evenodd\'%3E%3Cg fill=\'%23ffffff\' fill-opacity=\'0.03\'%3E%3Cpath d=\'M36 34v-4h-2v4h-4v2h4v4h2v-4h4v-2h-4zm0-30V0h-2v4h-4v2h4v4h2V6h4V4h-4zM6 34v-4H4v4H0v2h4v4h2v-4h4v-2H6zM6 4V0H4v4H0v2h4v4h2V6h4V4H6z\'/%3E%3C/g%3E%3C/g%3E%3C/svg%3E");\n}\n.hero-title {\n  font-size: 22px; font-weight: 800; color: white;\n  line-height: 1.2; margin-bottom: 4px; position: relative;\n}\n.hero-kannada {\n  font-family: \'Noto Sans Kannada\', sans-serif;\n  font-size: 15px; color: rgba(255,255,255,0.75);\n  margin-bottom: 16px; position: relative;\n}\n.hero-steps {\n  display: flex; justify-content: center; gap: 6px;\n  position: relative;\n}\n.hero-step {\n  background: rgba(255,255,255,0.12);\n  border: 1px solid rgba(255,255,255,0.2);\n  border-radius: 20px; padding: 5px 12px;\n  font-size: 11px; color: rgba(255,255,255,0.85);\n  display: flex; align-items: center; gap: 4px;\n}\n\n/* ── MAIN CARD ── */\n.card {\n  background: var(--white);\n  margin: 16px;\n  border-radius: 20px;\n  overflow: hidden;\n  box-shadow: 0 4px 20px rgba(0,0,0,0.08);\n  border: 1px solid var(--border);\n}\n.card-header {\n  background: var(--green-dark);\n  padding: 14px 18px;\n  display: flex; align-items: center; gap: 10px;\n}\n.card-header-icon { font-size: 22px; }\n.card-header-title { font-size: 16px; font-weight: 700; color: white; }\n.card-header-sub { font-size: 11px; color: rgba(255,255,255,0.6); font-family: \'Noto Sans Kannada\'; }\n.card-body { padding: 18px; }\n\n/* ── CROP SELECTOR ── */\n.crop-grid {\n  display: grid; grid-template-columns: 1fr 1fr 1fr;\n  gap: 8px; margin-bottom: 18px;\n}\n.crop-btn {\n  border: 2px solid var(--border);\n  border-radius: 14px; padding: 12px 6px;\n  background: var(--surface); cursor: pointer;\n  text-align: center; transition: all 0.18s;\n  font-family: \'Baloo 2\', \'Noto Sans Kannada\', sans-serif;\n}\n.crop-btn.active {\n  border-color: var(--green-main);\n  background: var(--green-pale);\n}\n.crop-btn-icon { font-size: 26px; display: block; margin-bottom: 3px; }\n.crop-btn-name { font-size: 12px; font-weight: 600; color: var(--text-dark); }\n.crop-btn-kannada { font-size: 10px; color: var(--text-light); font-family: \'Noto Sans Kannada\'; }\n\n/* ── UPLOAD ZONE ── */\n.upload-zone {\n  border: 2.5px dashed var(--green-light);\n  border-radius: 16px; min-height: 200px;\n  display: flex; flex-direction: column;\n  align-items: center; justify-content: center;\n  gap: 10px; cursor: pointer;\n  background: linear-gradient(135deg, #f0fbf0, #e8f5e8);\n  position: relative; overflow: hidden;\n  transition: all 0.2s;\n  -webkit-tap-highlight-color: transparent;\n}\n.upload-zone.has-image { min-height: 240px; border-style: solid; border-color: var(--green-main); }\n.upload-zone:active { background: #e0f5e0; transform: scale(0.99); }\n.upload-zone input[type="file"] {\n  position: absolute; inset: 0;\n  opacity: 0; cursor: pointer; font-size: 0;\n}\n.upload-icon { font-size: 48px; pointer-events: none; }\n.upload-text { font-size: 16px; font-weight: 700; color: var(--green-mid); pointer-events: none; }\n.upload-hint { font-size: 12px; color: var(--text-light); font-family: \'Noto Sans Kannada\'; pointer-events: none; }\n.preview-img {\n  width: 100%; height: 240px;\n  object-fit: cover; border-radius: 14px;\n  display: none;\n}\n.preview-overlay {\n  position: absolute; bottom: 8px; right: 8px;\n  background: rgba(0,0,0,0.55); color: white;\n  font-size: 11px; padding: 4px 10px; border-radius: 10px;\n  display: none;\n}\n\n/* ── CAMERA BUTTONS ── */\n.capture-row {\n  display: grid; grid-template-columns: 1fr 1fr;\n  gap: 10px; margin-top: 14px;\n}\n.btn-camera {\n  padding: 14px;\n  border: none; border-radius: 14px;\n  font-family: \'Baloo 2\', sans-serif;\n  font-size: 14px; font-weight: 700;\n  cursor: pointer; transition: all 0.18s;\n  display: flex; align-items: center; justify-content: center; gap: 6px;\n  -webkit-tap-highlight-color: transparent;\n}\n.btn-camera:active { transform: scale(0.97); }\n.btn-camera.primary {\n  background: var(--green-main);\n  color: white;\n  box-shadow: 0 4px 12px rgba(61,139,61,0.35);\n}\n.btn-camera.secondary {\n  background: var(--surface);\n  color: var(--green-mid);\n  border: 2px solid var(--border);\n}\n.btn-camera .cam-icon { font-size: 20px; }\n\n/* ── FARMER INFO ── */\n.input-row { margin-bottom: 12px; }\n.input-label {\n  font-size: 12px; font-weight: 600; color: var(--text-mid);\n  margin-bottom: 4px; display: block;\n}\n.input-label .label-kn { font-family: \'Noto Sans Kannada\'; font-weight: 400; color: var(--text-light); font-size: 11px; margin-left: 4px; }\n.input-field {\n  width: 100%; padding: 12px 14px;\n  border: 2px solid var(--border); border-radius: 12px;\n  font-family: \'Baloo 2\', \'Noto Sans Kannada\', sans-serif;\n  font-size: 15px; color: var(--text-dark);\n  background: var(--surface); outline: none;\n  transition: border-color 0.2s;\n  -webkit-appearance: none;\n}\n.input-field:focus { border-color: var(--green-main); background: white; }\n\n/* ── ANALYZE BUTTON ── */\n.btn-analyze {\n  width: 100%; padding: 18px;\n  background: linear-gradient(135deg, #3d8b3d, #2d6a2d);\n  border: none; border-radius: 16px;\n  font-family: \'Baloo 2\', sans-serif;\n  font-size: 18px; font-weight: 800;\n  color: white; cursor: pointer;\n  transition: all 0.2s;\n  box-shadow: 0 6px 20px rgba(45,106,45,0.4);\n  display: flex; align-items: center; justify-content: center; gap: 8px;\n  -webkit-tap-highlight-color: transparent;\n  margin-top: 4px;\n}\n.btn-analyze:active { transform: scale(0.98); box-shadow: 0 2px 8px rgba(45,106,45,0.3); }\n.btn-analyze:disabled { opacity: 0.5; cursor: not-allowed; }\n.btn-analyze .kn { font-family: \'Noto Sans Kannada\'; font-size: 14px; font-weight: 600; opacity: 0.85; }\n\n/* ── LOADING ── */\n.loading-card {\n  display: none;\n  margin: 0 16px 16px;\n  background: var(--green-dark);\n  border-radius: 20px; padding: 28px 20px;\n  text-align: center;\n  box-shadow: 0 4px 20px rgba(0,0,0,0.1);\n}\n.loading-spinner {\n  width: 48px; height: 48px;\n  border: 4px solid rgba(255,255,255,0.2);\n  border-top-color: #7ed87e;\n  border-radius: 50%;\n  animation: spin 0.8s linear infinite;\n  margin: 0 auto 14px;\n}\n@keyframes spin { to { transform: rotate(360deg); } }\n.loading-text { font-size: 16px; font-weight: 700; color: white; margin-bottom: 4px; }\n.loading-sub  { font-size: 12px; color: rgba(255,255,255,0.55); font-family: \'Noto Sans Kannada\'; }\n\n/* ── RESULT CARD ── */\n.result-card {\n  display: none;\n  margin: 0 16px 16px;\n  border-radius: 20px; overflow: hidden;\n  box-shadow: 0 4px 20px rgba(0,0,0,0.1);\n  animation: slideUp 0.4s cubic-bezier(.16,1,.3,1);\n}\n@keyframes slideUp { from { opacity:0; transform:translateY(20px) } to { opacity:1; transform:none } }\n\n.result-header {\n  padding: 20px 18px 16px;\n  display: flex; align-items: flex-start; gap: 14px;\n}\n.result-header.danger  { background: var(--red-light);    border-bottom: 3px solid var(--red); }\n.result-header.warning { background: var(--orange-light); border-bottom: 3px solid var(--orange); }\n.result-header.safe    { background: var(--green-pale);   border-bottom: 3px solid var(--green-main); }\n.result-icon { font-size: 40px; line-height: 1; flex-shrink: 0; }\n.result-disease { font-size: 20px; font-weight: 800; line-height: 1.2; }\n.result-header.danger  .result-disease { color: var(--red); }\n.result-header.warning .result-disease { color: var(--orange); }\n.result-header.safe    .result-disease { color: #1b5e20; }\n.result-confidence { font-size: 13px; font-weight: 600; color: var(--text-mid); margin-top: 3px; }\n.result-scan-id { font-size: 10px; color: var(--text-light); font-family: monospace; margin-top: 2px; }\n\n.conf-bar {\n  height: 6px; border-radius: 3px;\n  background: rgba(0,0,0,0.08); margin: 10px 0 0; overflow: hidden;\n}\n.conf-fill { height: 100%; border-radius: 3px; transition: width 1s cubic-bezier(.16,1,.3,1); width: 0; }\n.danger  .conf-fill { background: linear-gradient(90deg, var(--red), #e57373); }\n.warning .conf-fill { background: linear-gradient(90deg, var(--orange), #ffb74d); }\n.safe    .conf-fill { background: linear-gradient(90deg, var(--green-main), var(--green-light)); }\n\n.result-body { background: white; padding: 18px; }\n\n.kannada-box {\n  background: #e8f5e8; border: 1px solid var(--green-pale);\n  border-radius: 12px; padding: 14px 16px; margin-bottom: 16px;\n}\n.kannada-label { font-size: 10px; font-weight: 700; color: var(--green-mid); letter-spacing: 1px; text-transform: uppercase; margin-bottom: 4px; }\n.kannada-text {\n  font-family: \'Noto Sans Kannada\', sans-serif;\n  font-size: 15px; font-weight: 600; color: #1b5e20; line-height: 1.6;\n}\n\n.section-title {\n  font-size: 11px; font-weight: 700; color: var(--text-light);\n  text-transform: uppercase; letter-spacing: 1px;\n  margin-bottom: 10px; margin-top: 16px;\n  display: flex; align-items: center; gap: 6px;\n}\n.section-title:first-child { margin-top: 0; }\n\n.treatment-list { display: flex; flex-direction: column; gap: 8px; }\n.treatment-item {\n  display: flex; align-items: flex-start; gap: 10px;\n  background: var(--surface); border-radius: 10px; padding: 10px 12px;\n  border: 1px solid var(--border);\n}\n.treatment-num {\n  background: var(--green-main); color: white;\n  width: 22px; height: 22px; border-radius: 50%;\n  font-size: 11px; font-weight: 700;\n  display: flex; align-items: center; justify-content: center;\n  flex-shrink: 0; margin-top: 1px;\n}\n.treatment-text { font-size: 13px; color: var(--text-dark); line-height: 1.4; }\n\n.other-preds { display: flex; flex-direction: column; gap: 6px; }\n.pred-row { display: flex; align-items: center; gap: 8px; }\n.pred-name { font-size: 12px; color: var(--text-mid); width: 140px; flex-shrink: 0; }\n.pred-bar-track { flex: 1; height: 5px; background: var(--border); border-radius: 3px; overflow: hidden; }\n.pred-bar-fill  { height: 100%; background: var(--green-light); border-radius: 3px; }\n.pred-pct { font-size: 11px; color: var(--text-light); font-family: monospace; width: 36px; text-align: right; }\n\n.kvk-box {\n  background: var(--green-dark); border-radius: 14px;\n  padding: 14px 16px; margin-top: 16px;\n  display: flex; align-items: center; gap: 12px;\n}\n.kvk-icon { font-size: 28px; }\n.kvk-title { font-size: 13px; font-weight: 700; color: #7ed87e; }\n.kvk-number { font-size: 18px; font-weight: 800; color: white; letter-spacing: 1px; }\n.kvk-hint { font-size: 10px; color: rgba(255,255,255,0.5); font-family: \'Noto Sans Kannada\'; }\n\n.share-row { display: grid; grid-template-columns: 1fr 1fr; gap: 10px; margin-top: 14px; }\n.btn-share {\n  padding: 12px; border: none; border-radius: 12px;\n  font-family: \'Baloo 2\', sans-serif; font-size: 13px; font-weight: 700;\n  cursor: pointer; display: flex; align-items: center; justify-content: center; gap: 6px;\n  transition: all 0.18s; -webkit-tap-highlight-color: transparent;\n}\n.btn-share:active { transform: scale(0.97); }\n.btn-whatsapp { background: #25d366; color: white; }\n.btn-rescan   { background: var(--surface); color: var(--green-mid); border: 2px solid var(--border); }\n\n/* ── HISTORY ── */\n.history-card { background: white; margin: 0 16px 16px; border-radius: 20px; overflow: hidden; border: 1px solid var(--border); }\n.history-header { background: var(--surface); padding: 12px 16px; border-bottom: 1px solid var(--border); font-size: 13px; font-weight: 700; color: var(--text-mid); display: flex; align-items: center; gap: 6px; }\n.history-item { padding: 12px 16px; border-bottom: 1px solid #f0f7f0; display: flex; align-items: center; gap: 12px; }\n.history-item:last-child { border-bottom: none; }\n.history-dot { width: 10px; height: 10px; border-radius: 50%; flex-shrink: 0; }\n.history-dot.danger  { background: var(--red); }\n.history-dot.warning { background: var(--orange); }\n.history-dot.safe    { background: var(--green-main); }\n.history-disease { font-size: 13px; font-weight: 700; color: var(--text-dark); }\n.history-meta    { font-size: 11px; color: var(--text-light); }\n.history-conf    { margin-left: auto; font-size: 12px; font-weight: 700; font-family: monospace; color: var(--green-mid); }\n\n/* ── DEMO BADGE ── */\n.demo-notice {\n  margin: 0 16px 16px;\n  background: #fff8e1; border: 1px solid #ffe082;\n  border-radius: 12px; padding: 10px 14px;\n  font-size: 12px; color: #5d4037;\n  display: flex; align-items: center; gap: 8px;\n}\n\n/* ── NO-IMAGE STATE ── */\n.no-image-notice {\n  display: none;\n  background: #ffebee; border: 1px solid #ffcdd2;\n  border-radius: 10px; padding: 10px 14px;\n  font-size: 13px; color: var(--red);\n  margin-top: 10px;\n  text-align: center;\n}\n</style>\n</head>\n\n<body>\n<!-- TOP BAR -->\n<div class="topbar">\n  <div class="topbar-logo">🌾</div>\n  <div>\n    <div class="topbar-name">KrishiAI</div>\n    <div class="topbar-sub">ಬೆಳೆ ರೋಗ ಪತ್ತೆ ವ್ಯವಸ್ಥೆ</div>\n  </div>\n  <div class="topbar-right">\n    <div class="mode-pill" id="mode-pill">LOADING...</div>\n  </div>\n</div>\n\n<div class="scroll-wrap">\n\n  <!-- HERO -->\n  <div class="hero">\n    <div class="hero-title">📸 ಫೋಟೋ ತೆಗೆದು ರೋಗ ತಿಳಿಯಿರಿ</div>\n    <div class="hero-kannada">Take a photo of your crop leaf to detect disease</div>\n    <div class="hero-steps">\n      <div class="hero-step">📷 Photo</div>\n      <div class="hero-step">🤖 AI Scan</div>\n      <div class="hero-step">💊 Treatment</div>\n    </div>\n  </div>\n\n  <!-- DEMO NOTICE (shown when running without real model) -->\n  <div class="demo-notice" id="demo-notice" style="display:none">\n    ⚠️ Running in <strong>demo mode</strong> — train the AI model first (step2_train.py), then restart the server.\n  </div>\n\n  <!-- MAIN UPLOAD CARD -->\n  <div class="card">\n    <div class="card-header">\n      <span class="card-header-icon">🔬</span>\n      <div>\n        <div class="card-header-title">Scan Your Crop</div>\n        <div class="card-header-sub">ನಿಮ್ಮ ಬೆಳೆಯ ಎಲೆ ಫೋಟೋ ತೆಗೆಯಿರಿ</div>\n      </div>\n    </div>\n    <div class="card-body">\n\n      <!-- Crop type selector -->\n      <div style="margin-bottom:14px">\n        <div class="input-label">Crop Type <span class="label-kn">ಬೆಳೆ ಆಯ್ಕೆ</span></div>\n        <div class="crop-grid">\n          <button class="crop-btn active" onclick="selectCrop(this,\'paddy\')">\n            <span class="crop-btn-icon">🌾</span>\n            <span class="crop-btn-name">Paddy</span>\n            <span class="crop-btn-kannada">ಭತ್ತ</span>\n          </button>\n          <button class="crop-btn" onclick="selectCrop(this,\'coconut\')">\n            <span class="crop-btn-icon">🥥</span>\n            <span class="crop-btn-name">Coconut</span>\n            <span class="crop-btn-kannada">ತೆಂಗು</span>\n          </button>\n          <button class="crop-btn" onclick="selectCrop(this,\'areca\')">\n            <span class="crop-btn-icon">🌴</span>\n            <span class="crop-btn-name">Areca</span>\n            <span class="crop-btn-kannada">ಅಡಿಕೆ</span>\n          </button>\n        </div>\n      </div>\n\n      <!-- Upload zone -->\n      <div class="upload-zone" id="upload-zone">\n        <input type="file" accept="image/*" id="file-input" onchange="handleFile(event)">\n        <img class="preview-img" id="preview-img">\n        <div class="preview-overlay" id="preview-overlay">📸 Tap to change</div>\n        <span class="upload-icon" id="up-icon">🌿</span>\n        <div class="upload-text" id="up-text">Tap to select a photo</div>\n        <div class="upload-hint" id="up-hint">ಫೋಟೋ ಆಯ್ಕೆ ಮಾಡಿ ಅಥವಾ ತೆಗೆಯಿರಿ</div>\n      </div>\n\n      <!-- Camera / Gallery buttons -->\n      <div class="capture-row">\n        <button class="btn-camera primary" onclick="openCamera()">\n          <span class="cam-icon">📷</span> Camera\n        </button>\n        <button class="btn-camera secondary" onclick="openGallery()">\n          <span class="cam-icon">🖼️</span> Gallery\n        </button>\n      </div>\n\n      <!-- Hidden camera input -->\n      <input type="file" accept="image/*" capture="environment" id="camera-input" style="display:none" onchange="handleFile(event)">\n      <input type="file" accept="image/*"                        id="gallery-input" style="display:none" onchange="handleFile(event)">\n\n      <!-- No-image warning -->\n      <div class="no-image-notice" id="no-image-notice">\n        ⚠️ Please take or select a photo first! <br>\n        <span style="font-family:\'Noto Sans Kannada\'">ಮೊದಲು ಫೋಟೋ ತೆಗೆಯಿರಿ</span>\n      </div>\n\n      <!-- Farmer info -->\n      <div style="margin-top:18px; border-top:1px solid var(--border); padding-top:16px">\n        <div class="input-row">\n          <label class="input-label" for="farmer-name">\n            Your Name (optional) <span class="label-kn">ನಿಮ್ಮ ಹೆಸರು</span>\n          </label>\n          <input class="input-field" type="text" id="farmer-name"\n                 placeholder="ರಾಮಣ್ಣ ಗೌಡ / Ramanna Gowda"\n                 autocomplete="name">\n        </div>\n        <div class="input-row">\n          <label class="input-label" for="plot-name">\n            Plot / Village <span class="label-kn">ಜಮೀನು / ಊರು</span>\n          </label>\n          <input class="input-field" type="text" id="plot-name"\n                 placeholder="e.g. Manipal Road Plot A1"\n                 autocomplete="off">\n        </div>\n      </div>\n\n      <!-- Analyze button -->\n      <button class="btn-analyze" id="analyze-btn" onclick="runAnalysis()">\n        🔍 Analyze Crop &nbsp;<span class="kn">ರೋಗ ಪತ್ತೆ ಮಾಡಿ</span>\n      </button>\n\n    </div>\n  </div>\n\n  <!-- LOADING STATE -->\n  <div class="loading-card" id="loading-card">\n    <div class="loading-spinner"></div>\n    <div class="loading-text">Analyzing your crop...</div>\n    <div class="loading-sub">ನಿಮ್ಮ ಬೆಳೆ ತಪಾಸಣೆ ನಡೆಯುತ್ತಿದೆ...</div>\n  </div>\n\n  <!-- RESULT CARD -->\n  <div class="result-card" id="result-card">\n    <!-- header injected by JS -->\n  </div>\n\n  <!-- HISTORY -->\n  <div class="history-card" id="history-card" style="display:none">\n    <div class="history-header">📋 Recent Scans &nbsp;<span style="font-family:\'Noto Sans Kannada\';font-weight:400;font-size:12px">ಇತ್ತೀಚಿನ ತಪಾಸಣೆ</span></div>\n    <div id="history-list"></div>\n  </div>\n\n</div><!-- /scroll-wrap -->\n\n<script>\n// ── State ──\nlet selectedCrop    = \'paddy\';\nlet selectedFile    = null;\nlet scanHistory     = [];\n\n// ── Check server mode on load ──\nfetch(\'/health\')\n  .then(r => r.json())\n  .then(d => {\n    const pill = document.getElementById(\'mode-pill\');\n    if (d.model_ready) {\n      pill.textContent = \'LIVE AI\';\n      pill.style.background = \'rgba(90,175,90,0.2)\';\n      pill.style.color = \'#7ed87e\';\n    } else {\n      pill.textContent = \'DEMO\';\n      pill.style.background = \'rgba(244,168,0,0.2)\';\n      pill.style.color = \'#f4a800\';\n      pill.style.borderColor = \'rgba(244,168,0,0.3)\';\n      document.getElementById(\'demo-notice\').style.display = \'flex\';\n    }\n  })\n  .catch(() => {\n    document.getElementById(\'mode-pill\').textContent = \'OFFLINE\';\n  });\n\n// ── Crop selection ──\nfunction selectCrop(btn, crop) {\n  selectedCrop = crop;\n  document.querySelectorAll(\'.crop-btn\').forEach(b => b.classList.remove(\'active\'));\n  btn.classList.add(\'active\');\n}\n\n// ── Camera / Gallery ──\nfunction openCamera()  { document.getElementById(\'camera-input\').click(); }\nfunction openGallery() { document.getElementById(\'gallery-input\').click(); }\n\nfunction handleFile(e) {\n  const file = e.target.files[0];\n  if (!file) return;\n  selectedFile = file;\n  const url    = URL.createObjectURL(file);\n  const preview = document.getElementById(\'preview-img\');\n  const zone    = document.getElementById(\'upload-zone\');\n\n  preview.src = url;\n  preview.style.display = \'block\';\n  document.getElementById(\'preview-overlay\').style.display = \'block\';\n  document.getElementById(\'up-icon\').style.display = \'none\';\n  document.getElementById(\'up-text\').style.display = \'none\';\n  document.getElementById(\'up-hint\').style.display = \'none\';\n  zone.classList.add(\'has-image\');\n  document.getElementById(\'no-image-notice\').style.display = \'none\';\n  // Reset any previous result\n  document.getElementById(\'result-card\').style.display = \'none\';\n}\n\n// Also allow tapping the zone to open file picker\ndocument.getElementById(\'upload-zone\').addEventListener(\'click\', function(e) {\n  if (e.target.tagName === \'INPUT\') return;\n  document.getElementById(\'file-input\').click();\n});\n\n// ── Main analysis ──\nasync function runAnalysis() {\n  if (!selectedFile) {\n    document.getElementById(\'no-image-notice\').style.display = \'block\';\n    document.getElementById(\'no-image-notice\').scrollIntoView({ behavior: \'smooth\', block: \'center\' });\n    return;\n  }\n\n  // Show loading\n  document.getElementById(\'loading-card\').style.display = \'block\';\n  document.getElementById(\'result-card\').style.display  = \'none\';\n  document.getElementById(\'analyze-btn\').disabled = true;\n  document.getElementById(\'loading-card\').scrollIntoView({ behavior: \'smooth\', block: \'center\' });\n\n  try {\n    const formData = new FormData();\n    formData.append(\'image\',       selectedFile);\n    formData.append(\'farmer_name\', document.getElementById(\'farmer-name\').value.trim());\n    formData.append(\'crop_type\',   selectedCrop);\n    formData.append(\'plot_name\',   document.getElementById(\'plot-name\').value.trim());\n\n    const resp   = await fetch(\'/predict\', { method: \'POST\', body: formData });\n    const result = await resp.json();\n\n    if (!resp.ok) {\n      throw new Error(result.error || \'Server error\');\n    }\n\n    showResult(result);\n    addToHistory(result);\n\n  } catch (err) {\n    document.getElementById(\'loading-card\').style.display = \'none\';\n    alert(\'Error: \' + err.message + \'\\n\\nMake sure the Flask server is running.\');\n  } finally {\n    document.getElementById(\'analyze-btn\').disabled = false;\n  }\n}\n\n// ── Render result ──\nfunction showResult(r) {\n  document.getElementById(\'loading-card\').style.display = \'none\';\n\n  const isHealthy = r.risk_level === \'LOW\';\n  const isHigh    = r.risk_level === \'HIGH\';\n  const cls       = isHealthy ? \'safe\' : isHigh ? \'danger\' : \'warning\';\n  const icon      = isHealthy ? \'✅\' : isHigh ? \'🚨\' : \'⚠️\';\n  const confPct   = parseFloat(r.confidence_pct) || (r.confidence * 100);\n\n  // Treatment items\n  const treatments = (r.treatment || []).map((t, i) => `\n    <div class="treatment-item">\n      <div class="treatment-num">${i+1}</div>\n      <div class="treatment-text">${t}</div>\n    </div>\n  `).join(\'\');\n\n  // Top-k predictions\n  const topK = (r.top_k || []).map(p => `\n    <div class="pred-row">\n      <div class="pred-name">${p.disease}</div>\n      <div class="pred-bar-track"><div class="pred-bar-fill" style="width:${(p.confidence*100).toFixed(0)}%"></div></div>\n      <div class="pred-pct">${(p.confidence*100).toFixed(0)}%</div>\n    </div>\n  `).join(\'\');\n\n  // WhatsApp message\n  const waMsg = encodeURIComponent(\n    `🌾 *KrishiAI Diagnosis*\\n\\n` +\n    `${icon} *${r.disease}*\\n` +\n    `Confidence: ${r.confidence_pct}\\n` +\n    `Risk: ${r.risk_level}\\n` +\n    `Scan ID: ${r.scan_id}\\n\\n` +\n    `📋 *Treatment:*\\n` +\n    (r.treatment||[]).map(t => `• ${t}`).join(\'\\n\') + `\\n\\n` +\n    `🇮🇳 ${r.kannada_alert}\\n\\n` +\n    `_KVK Udupi: 0820-2520842_`\n  );\n\n  const card = document.getElementById(\'result-card\');\n  card.innerHTML = `\n    <div class="result-header ${cls}">\n      <div class="result-icon">${icon}</div>\n      <div style="flex:1">\n        <div class="result-disease">${r.disease}</div>\n        <div class="result-confidence">\n          Confidence: ${r.confidence_pct} &nbsp;·&nbsp; Risk: ${r.risk_level}\n        </div>\n        <div class="result-scan-id">Scan #${r.scan_id} · ${r.timestamp}</div>\n        <div class="conf-bar ${cls}">\n          <div class="conf-fill" id="cf" style="width:0%"></div>\n        </div>\n      </div>\n    </div>\n    <div class="result-body">\n\n      <div class="kannada-box">\n        <div class="kannada-label">ಕನ್ನಡ ಸಂದೇಶ · Kannada Alert</div>\n        <div class="kannada-text">${r.kannada_alert}</div>\n      </div>\n\n      <div class="section-title">💊 Treatment Recommendations</div>\n      <div class="treatment-list">${treatments}</div>\n\n      ${topK ? `<div class="section-title" style="margin-top:16px">📊 Model Confidence Breakdown</div>\n      <div class="other-preds">${topK}</div>` : \'\'}\n\n      <div class="kvk-box">\n        <div class="kvk-icon">📞</div>\n        <div>\n          <div class="kvk-title">KVK Udupi — Field Officer</div>\n          <div class="kvk-number">0820-2520842</div>\n          <div class="kvk-hint">ಸಹಾಯಕ್ಕಾಗಿ ಸಂಪರ್ಕಿಸಿ · Call for expert help</div>\n        </div>\n      </div>\n\n      ${r.demo_mode ? `<div style="margin-top:12px;font-size:11px;color:#9e9e9e;text-align:center">Demo mode — results are simulated</div>` : \'\'}\n\n      <div class="share-row">\n        <a href="https://wa.me/?text=${waMsg}" target="_blank" style="text-decoration:none">\n          <button class="btn-share btn-whatsapp" style="width:100%">\n            💬 Share on WhatsApp\n          </button>\n        </a>\n        <button class="btn-share btn-rescan" onclick="resetScan()">\n          🔄 Scan Again<br><small style="font-family:\'Noto Sans Kannada\'">ಮತ್ತೆ ತಪಾಸಿಸಿ</small>\n        </button>\n      </div>\n\n    </div>\n  `;\n\n  card.style.display = \'block\';\n  card.scrollIntoView({ behavior: \'smooth\', block: \'start\' });\n\n  // Animate confidence bar\n  setTimeout(() => {\n    const cf = document.getElementById(\'cf\');\n    if (cf) cf.style.width = confPct.toFixed(0) + \'%\';\n  }, 200);\n}\n\n// ── History ──\nfunction addToHistory(r) {\n  const isHealthy = r.risk_level === \'LOW\';\n  const isHigh    = r.risk_level === \'HIGH\';\n  const dotCls    = isHealthy ? \'safe\' : isHigh ? \'danger\' : \'warning\';\n\n  scanHistory.unshift({\n    disease:    r.disease,\n    conf:       r.confidence_pct,\n    time:       r.timestamp,\n    risk:       r.risk_level,\n    dot:        dotCls,\n  });\n\n  renderHistory();\n  document.getElementById(\'history-card\').style.display = \'block\';\n}\n\nfunction renderHistory() {\n  const list = document.getElementById(\'history-list\');\n  list.innerHTML = scanHistory.slice(0, 8).map(h => `\n    <div class="history-item">\n      <div class="history-dot ${h.dot}"></div>\n      <div>\n        <div class="history-disease">${h.disease}</div>\n        <div class="history-meta">${h.time}</div>\n      </div>\n      <div class="history-conf">${h.conf}</div>\n    </div>\n  `).join(\'\');\n}\n\n// ── Reset ──\nfunction resetScan() {\n  selectedFile = null;\n  document.getElementById(\'preview-img\').style.display      = \'none\';\n  document.getElementById(\'preview-overlay\').style.display  = \'none\';\n  document.getElementById(\'up-icon\').style.display          = \'\';\n  document.getElementById(\'up-text\').style.display          = \'\';\n  document.getElementById(\'up-hint\').style.display          = \'\';\n  document.getElementById(\'upload-zone\').classList.remove(\'has-image\');\n  document.getElementById(\'result-card\').style.display      = \'none\';\n  document.getElementById(\'file-input\').value               = \'\';\n  document.getElementById(\'camera-input\').value             = \'\';\n  document.getElementById(\'gallery-input\').value            = \'\';\n  window.scrollTo({ top: 0, behavior: \'smooth\' });\n}\n</script>\n</body>\n</html>\n"""\n\n\n# ──────────────────────────────────────────────────────────────────────────────\n# RUN\n# ──────────────────────────────────────────────────────────────────────────────\n\nif __name__ == "__main__":\n    import socket\n\n    # Find local IP so farmers on the same WiFi can connect\n    try:\n        local_ip = socket.gethostbyname(socket.gethostname())\n    except Exception:\n        local_ip = "localhost"\n\n    print("╔══════════════════════════════════════════════════════════╗")\n    print("║           KrishiAI — Farmer Web Server                  ║")\n    print("╚══════════════════════════════════════════════════════════╝")\n    print()\n    print(f"  Model status : {\'✅ Live AI\' if MODEL_READY else \'⚠️  Demo mode (run step2_train.py first)\'}")\n    print()\n    print(f"  Local access   :  http://localhost:5000")\n    print(f"  On same WiFi   :  http://{local_ip}:5000")\n    print()\n    print(f"  For internet access (public URL):")\n    print(f"    pip install pyngrok")\n    print(f"    ngrok http 5000")\n    print()\n    print(f"  Farmer opens the URL on their phone → takes a photo → gets diagnosis")\n    print()\n\n    app.run(host="0.0.0.0", port=5000, debug=False)\n'

app_path = os.path.join(BASE_DIR, "app.py")
with open(app_path, "w", encoding="utf-8") as f:
    f.write(app_code)
print(f"✅ app.py written → {app_path}")
print(f"   Size: {os.path.getsize(app_path)/1e3:.0f} KB  |  {len(app_code.splitlines())} lines")

## ▶️ Option A — Run Locally (same WiFi network)

In [ ]:
# Print instructions for local run
import socket
try:
    local_ip = socket.gethostbyname(socket.gethostname())
except:
    local_ip = "localhost"

print("To start the server, open a terminal and run:")
print()
print(f"  cd {BASE_DIR}")
print(f"  python app.py")
print()
print("Then open on any phone connected to the same WiFi:")
print(f"  http://{local_ip}:5000")
print()
print("Farmer opens this URL in their phone browser.")
print("No app install required — works on any Android/iOS browser.")

## ▶️ Option B — Public URL via ngrok (Colab / remote access)

In [ ]:
# ── Uncomment to launch from inside this notebook (Colab recommended) ────────

# !pip install pyngrok -q
# from pyngrok import ngrok
# import threading, subprocess, time

# def run_flask():
#     subprocess.run(["python", os.path.join(BASE_DIR, "app.py")])

# threading.Thread(target=run_flask, daemon=True).start()
# time.sleep(4)   # wait for server to boot

# public_url = ngrok.connect(5000)
# print(f"\n🌐 Farmer App Public URL: {public_url}")
# print("   Share this URL — farmers open it on their phone anywhere in India!")

print("Uncomment the block above to get a live public URL from Google Colab.")

## 📱 What the Farmer Sees

The web app has these features:

In [ ]:
features = {
    "Crop selector"        : "Paddy / Coconut / Areca buttons",
    "Camera capture"       : "Opens phone camera directly (capture=environment)",
    "Gallery upload"       : "Pick from existing photos",
    "Farmer name / plot"   : "Optional fields for scan logging",
    "AI diagnosis"         : "Disease name + confidence % from TFLite model",
    "Kannada alert"        : "Native language message displayed prominently",
    "Treatment steps"      : "Numbered, actionable steps (e.g. Tricyclazole 75WP)",
    "WhatsApp share"       : "One-tap pre-filled message to forward to KVK officer",
    "Scan history"         : "Last 8 scans shown on the same page",
    "Demo mode"            : "Works even before model is trained (cycles sample results)",
    "Health endpoint"      : "GET /health — shows if live AI or demo mode",
    "History endpoint"     : "GET /history — last 50 scans as JSON",
}
print("Farmer Web App Features:")
print()
for k, v in features.items():
    print(f"  ✅ {k:<22} — {v}")
print()
print("API routes:")
print("  GET  /          → Mobile web app (farmer.html embedded)")
print("  POST /predict   → Accept image → return JSON diagnosis")
print("  GET  /history   → Last 50 scans")
print("  GET  /health    → Server status + model_ready flag")

## 🧪 Test the API Directly (without opening the browser)

In [ ]:
# Test /predict endpoint using Python requests (server must be running first)
# Uncomment after starting app.py in a separate terminal or via ngrok above

# import requests
# test_df  = __import__("pandas").read_csv(os.path.join(LOGS_DIR, "test_manifest.csv"))
# img_path = test_df.iloc[0]["path"]

# with open(img_path, "rb") as f:
#     response = requests.post(
#         "http://localhost:5000/predict",
#         files={"image": ("leaf.jpg", f, "image/jpeg")},
#         data={"farmer_name": "Test Farmer", "crop_type": "paddy"}
#     )

# result = response.json()
# print(f"Scan ID    : {result['scan_id']}")
# print(f"Disease    : {result['disease']}")
# print(f"Confidence : {result['confidence_pct']}")
# print(f"Risk       : {result['risk_level']}")
# print(f"Kannada    : {result['kannada_alert']}")

print("Uncomment the block above after starting the server.")

In [ ]:
print("✅ app.py is ready!")
print()
print("Summary of all project outputs:")
checks = {
    "data/cleaned/"                         : DATA_CLEAN_DIR,
    "logs/train_manifest.csv"               : os.path.join(LOGS_DIR, "train_manifest.csv"),
    "logs/class_weights.json"               : os.path.join(LOGS_DIR, "class_weights.json"),
    "models/krishiai_final.keras"           : os.path.join(MODELS_DIR, "krishiai_final.keras"),
    "plots/phase1_training_curves.png"      : os.path.join(PLOTS_DIR, "phase1_training_curves.png"),
    "plots/confusion_matrix.png"            : os.path.join(PLOTS_DIR, "confusion_matrix.png"),
    "plots/gradcam_visualization.png"       : os.path.join(PLOTS_DIR, "gradcam_visualization.png"),
    f"tflite/{TFLITE_MODEL_NAME}"           : os.path.join(TFLITE_DIR, TFLITE_MODEL_NAME),
    "logs/evaluation_summary.json"          : os.path.join(LOGS_DIR, "evaluation_summary.json"),
    "app.py"                                : os.path.join(BASE_DIR, "app.py"),
}
for label, path in checks.items():
    exists = os.path.exists(path)
    size   = f"  ({os.path.getsize(path)/1e6:.1f} MB)" if exists and os.path.isfile(path) else ""
    print(f"  {'✅' if exists else '⬜'} {label}{size}")

summary_path = os.path.join(LOGS_DIR, "evaluation_summary.json")
if os.path.exists(summary_path):
    import json as _j
    s = _j.load(open(summary_path))
    print()
    print(f"  Model accuracy : {s['test_accuracy']*100:.2f}%  (target ≥ 90%)")
    print(f"  Macro F1       : {s['macro_f1']:.4f}")
    print(f"  Target met?    : {'✅ YES' if s['meets_target'] else '⚠️ Not yet'}")